In [13]:
!pip install -q pypdf chromadb sentence-transformers langchain-text-splitters

In [14]:
import os, shutil

os.makedirs("quickkart_corpus", exist_ok=True)

for name in os.listdir("."):
    if name.endswith((".txt", ".pdf")):
        shutil.move(name, os.path.join("quickkart_corpus", name))

print(sorted(os.listdir("quickkart_corpus")))

['cancellation_policy.txt', 'delivery_fees_and_slots.pdf', 'item_issues_policy.txt', 'late_delivery_policy.txt', 'payments_and_wallet.pdf', 'refund_policy.txt', 'special_cases_and_account_rules.pdf']


In [15]:
from pypdf import PdfReader

os.makedirs("quickkart_corpus", exist_ok=True)
for name in os.listdir("."):
    if name.endswith((".txt", ".pdf")):
        shutil.move(name, os.path.join("quickkart_corpus", name))

folder = "quickkart_corpus"
for name in sorted(os.listdir(folder)):
    path = os.path.join(folder, name)

    if name.endswith(".txt"):
        with open(path, encoding="utf-8") as f:
            text = f.read()
    elif name.endswith(".pdf"):
        text = ""
        for page in PdfReader(path).pages:
            text += page.extract_text()
    else:
        continue

    print(name, "->", len(text), "characters")
    print(text[:150])
    print("-----")

cancellation_policy.txt -> 2026 characters
QUICKKART CANCELLATION POLICY
Version 2.4 | Effective 1 September 2026
(Fictional policy written for a RAG learning project. QuickKart is not a real c
-----
delivery_fees_and_slots.pdf -> 1279 characters
QUICKKART DELIVERY FEES AND SLOTS
Version 3.1 | Effective 1 September 2026
(Fictional policy written for a RAG learning project. QuickKart is not a re
-----
item_issues_policy.txt -> 1979 characters
QUICKKART ITEM ISSUES POLICY
Version 1.9 | Effective 1 September 2026
(Fictional policy written for a RAG learning project. QuickKart is not a real co
-----
late_delivery_policy.txt -> 2451 characters
QUICKKART LATE DELIVERY POLICY
Version 2.7 | Effective 1 September 2026
(Fictional policy written for a RAG learning project. QuickKart is not a real 
-----
payments_and_wallet.pdf -> 1961 characters
QUICKKART PAYMENTS AND WALLET
Version 4.0 | Effective 1 September 2026
(Fictional policy written for a RAG learning project. QuickKart is not a real c
--

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = {}
for name in sorted(os.listdir("quickkart_corpus")):
    path = os.path.join("quickkart_corpus", name)
    if name.endswith(".txt"):
        with open(path, encoding="utf-8") as f:
            docs[name] = f.read()
    elif name.endswith(".pdf"):
        docs[name] = "\n".join(page.extract_text() for page in PdfReader(path).pages)

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = []
for name, text in docs.items():
    for piece in splitter.split_text(text):
        chunks.append({"text": piece, "source": name})

print("Total chunks:", len(chunks))
for name in docs:
    count = sum(1 for c in chunks if c["source"] == name)
    print(name, "->", count, "chunks")

print()
print(chunks[10])

Total chunks: 42
cancellation_policy.txt -> 6 chunks
delivery_fees_and_slots.pdf -> 3 chunks
item_issues_policy.txt -> 6 chunks
late_delivery_policy.txt -> 7 chunks
payments_and_wallet.pdf -> 6 chunks
refund_policy.txt -> 10 chunks
special_cases_and_account_rules.pdf -> 4 chunks

{'text': 'SECTION 1: TYPES OF ITEM ISSUES\n1.1 Missing item: an item you ordered and paid for is not in the delivery.\n1.2 Wrong item: you received a different product, variant or size from the one you ordered.\n1.3 Damaged item: the item or its packaging is broken, crushed or leaking.\n1.4 Spoiled or poor-quality item: for example rotten fruit, sour milk or wilted vegetables.', 'source': 'item_issues_policy.txt'}


In [17]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection("quickkart")

collection.add(
    ids=[f"chunk{i}" for i in range(len(chunks))],
    documents=[c["text"] for c in chunks],
    metadatas=[{"source": c["source"]} for c in chunks],
)

print("Stored:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 33.6MiB/s]


Stored: 42


In [18]:
def ask(question, n=3):
    results = collection.query(query_texts=[question], n_results=n)
    for i in range(n):
        source = results["metadatas"][0][i]["source"]
        distance = results["distances"][0][i]
        print(f"#{i+1} | {source} | distance {distance:.3f}")
        print(results["documents"][0][i])
        print()

ask("I received a yogurt that was already past its expiry date. What do I get and what do I need to do?")

#1 | item_issues_policy.txt | distance 1.162
SECTION 3: EVIDENCE REQUIRED
3.1 Missing items: no photo is needed. QuickKart checks the packing record instead.
3.2 Wrong, damaged and spoiled items: one clear photo of the item and its packaging or label is required.
3.3 Expired items: send a photo that clearly shows the expiry date on the label.
3.4 If a photo is unclear, support may ask for one more photo within 12 hours of your report.

#2 | item_issues_policy.txt | distance 1.252
1.5 Expired item: an item delivered after the expiry date printed on its label. You receive a full refund plus a Rs. 50 QuickKart Wallet credit.

#3 | refund_policy.txt | distance 1.284
SECTION 4: RETURNS AND NON-REFUNDABLE SITUATIONS
4.1 Change of mind: perishable products (fruits, vegetables, dairy, bakery, meat and frozen food) cannot be returned or refunded because you changed your mind. Sealed, unused non-perishable products in original packaging can be returned within 7 days of delivery. A return pickup 

In [19]:
import pandas as pd

key = pd.read_csv("answer_key.csv")
key = key[key["source_file"] != "NONE"]

key.loc[key["id"] == 13, "must_contain"] += "|Perishable items: report within 24 hours"

def found(chunk_texts, needed):
    text = " ".join(" ".join(chunk_texts).split())
    return all(phrase in text for phrase in needed)

def evaluate(k=3):
    top1 = topk = 0
    for _, row in key.iterrows():
        results = collection.query(query_texts=[row["question"]], n_results=k)
        retrieved = results["documents"][0]
        needed = row["must_contain"].split("|")
        top1 += found(retrieved[:1], needed)
        topk += found(retrieved, needed)
        if not found(retrieved, needed):
            print("MISS in top", k, ": Q", row["id"])
    print(f"Top-1: {top1}/{len(key)}   Top-{k}: {topk}/{len(key)}")

evaluate()

MISS in top 3 : Q 13
MISS in top 3 : Q 17
Top-1: 9/17   Top-3: 15/17


In [20]:
def build_collection(chunk_size, overlap):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    pieces = []
    for name, text in docs.items():
        for p in splitter.split_text(text):
            pieces.append({"text": p, "source": name})
    col = client.get_or_create_collection(f"quickkart_{chunk_size}_{overlap}")
    if col.count() == 0:
        col.add(
            ids=[f"chunk{i}" for i in range(len(pieces))],
            documents=[p["text"] for p in pieces],
            metadatas=[{"source": p["source"]} for p in pieces],
        )
    return col, len(pieces)

def evaluate(col, k=3):
    top1 = topk = 0
    missed = []
    for _, row in key.iterrows():
        results = col.query(query_texts=[row["question"]], n_results=k)
        retrieved = results["documents"][0]
        needed = row["must_contain"].split("|")
        top1 += found(retrieved[:1], needed)
        ok = found(retrieved, needed)
        topk += ok
        if not ok:
            missed.append(int(row["id"]))
    return top1, topk, missed

for size, overlap in [(200, 20), (200, 50), (500, 50), (500, 100), (1000, 100)]:
    col, n = build_collection(size, overlap)
    t1, tk, missed = evaluate(col)
    print(f"size={size} overlap={overlap} chunks={n} | top-1 {t1}/{len(key)} | top-3 {tk}/{len(key)} | missed {missed}")

size=200 overlap=20 chunks=106 | top-1 6/17 | top-3 10/17 | missed [2, 3, 6, 7, 9, 10, 13]
size=200 overlap=50 chunks=108 | top-1 6/17 | top-3 11/17 | missed [3, 6, 7, 9, 10, 13]
size=500 overlap=50 chunks=42 | top-1 9/17 | top-3 15/17 | missed [13, 17]
size=500 overlap=100 chunks=43 | top-1 8/17 | top-3 16/17 | missed [13]
size=1000 overlap=100 chunks=22 | top-1 10/17 | top-3 16/17 | missed [8]


In [21]:
def show(col_name, qid, k=3):
    col = client.get_collection(col_name)
    row = key[key["id"] == qid].iloc[0]
    print("Q", qid, "|", row["question"])
    print("Needs:", row["must_contain"])
    print()
    res = col.query(query_texts=[row["question"]], n_results=k)
    for i in range(k):
        print(f"#{i+1} | {res['metadatas'][0][i]['source']} | distance {res['distances'][0][i]:.3f}")
        print(res["documents"][0][i])
        print()

show("quickkart_500_100", 13)
show("quickkart_1000_100", 8)

Q 13 | I received a yogurt that was already past its expiry date. What do I get and what do I need to do?
Needs: full refund plus a Rs. 50|expiry date on the label|Perishable items: report within 24 hours

#1 | item_issues_policy.txt | distance 0.990
1.4 Spoiled or poor-quality item: for example rotten fruit, sour milk or wilted vegetables.
1.5 Expired item: an item delivered after the expiry date printed on its label. You receive a full refund plus a Rs. 50 QuickKart Wallet credit.

#2 | item_issues_policy.txt | distance 1.162
SECTION 3: EVIDENCE REQUIRED
3.1 Missing items: no photo is needed. QuickKart checks the packing record instead.
3.2 Wrong, damaged and spoiled items: one clear photo of the item and its packaging or label is required.
3.3 Expired items: send a photo that clearly shows the expiry date on the label.
3.4 If a photo is unclear, support may ask for one more photo within 12 hours of your report.

#3 | refund_policy.txt | distance 1.284
SECTION 4: RETURNS AND NON-REFU

In [22]:
!pip install -q groq

In [23]:
from groq import Groq
from google.colab import userdata
import time

groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-20b"

def answer(question, k=5):
    res = col.query(query_texts=[question], n_results=k)
    context = ""
    for text, meta in zip(res["documents"][0], res["metadatas"][0]):
        context += f"[Source: {meta['source']}]\n{text}\n\n"

    prompt = f"""You are a customer support assistant for QuickKart.
Answer the question using ONLY the policy excerpts below.
- Mention the source file name(s) you used.
- If the excerpts do not contain the answer, reply exactly: "I don't know based on the policy documents."
- Do not guess or use outside knowledge.
- If a rule depends on a time, amount or condition, compare it with the details in the question and say which case applies, with the number for that case.
- If the answer depends on a condition the question does not mention, state that condition.

POLICY EXCERPTS:
{context}
QUESTION: {question}"""

    for attempt in range(3):
        try:
            response = groq_client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            return response.choices[0].message.content
        except Exception as e:
            if "429" in str(e):
                return "RATE LIMITED: " + str(e)[:200]
            print(f"Attempt {attempt+1} failed: {str(e)[:80]}. Waiting 5 seconds...")
            time.sleep(5)
    return "FAILED after 3 attempts"

print(answer("One item was missing from my order. How much refund do I get?"))

You will receive a **full refund of the price of the missing item, including the tax that was charged on that item**.  

*Source: refund_policy.txt*


In [24]:
print(answer("I cancelled my order 10 minutes after placing it, before the delivery partner picked it up. What will I pay?"))
print("-----")
print(answer("I received a yogurt that was already past its expiry date. What do I get and what do I need to do?"))
print("-----")
print(answer("Can I change my delivery address after I have placed an order?"))
print("-----")
print(answer("Does QuickKart sell alcohol or cigarettes?"))

You will be charged a cancellation fee of **Rs. 20**.  
*Source: cancellation_policy.txt – Section 1, clauses 1.1 and 1.2.*
-----
**What you’ll receive**  
- A full refund of the yogurt’s price (including tax) **plus** a Rs. 50 QuickKart Wallet credit.  
  *Source: item_issues_policy.txt – Section 1.5; refund_policy.txt – Section 2.3.*

**What you need to do**  
1. **Report the issue within 24 hours** of delivery (yogurt is a perishable item).  
   *Source: item_issues_policy.txt – Section 2.1.*  
2. **Send a clear photo** that shows the expiry date on the yogurt’s label.  
   *Source: item_issues_policy.txt – Section 3.3.*  

Once you submit the report and photo within the 24‑hour window, QuickKart will process the refund and credit.
-----
I don't know based on the policy documents.
-----
I don't know based on the policy documents.


In [25]:
import time
import pandas as pd

key_df = pd.read_csv("answer_key.csv")
results = []

for _, row in key_df.iterrows():
    reply = answer(row["question"])
    if reply.startswith("RATE LIMITED"):
        print("Stopped at question", row["id"], "-", reply)
        break
    results.append({
        "id": row["id"],
        "question": row["question"],
        "type": row["type"],
        "expected": row["expected_answer"],
        "model": MODEL,
        "model_answer": reply,
    })
    time.sleep(4)

df = pd.DataFrame(results)
df.to_csv("generation_results_v2.csv", index=False)

for _, r in df.iterrows():
    print("Q", r["id"], "|", r["question"])
    print("EXPECTED:", r["expected"])
    print("MODEL   :", r["model_answer"])
    print("-----")

Q 1 | One item was missing from my order. How much refund do I get?
EXPECTED: A full refund of the missing item's price, including the tax charged on that item.
MODEL   : You will receive a **full refund of the price of the missing item, including the tax that was charged on that item**.  

*Source: refund_policy.txt*
-----
Q 2 | I paid by UPI. How long will my refund take?
EXPECTED: 2 to 3 working days (working days = Monday to Saturday, excluding public holidays).
MODEL   : The refund will be processed within **2 to 3 working days** for a UPI payment.  
(See *refund_policy.txt*, Section 3.2.)
-----
Q 3 | I paid by credit card. How long will the refund take, and is there a faster option?
EXPECTED: 5 to 7 working days to the card. Faster option: choose QuickKart Wallet credit, which arrives within 5 minutes, but the choice cannot be reversed.
MODEL   : **Refund timeline for a credit‑card payment**  
- According to the Refund Policy (refund_policy.txt, §3.2), a refund to a credit or deb

In [26]:
import sqlite3, random
from datetime import datetime, timedelta

random.seed(42)
conn = sqlite3.connect("quickkart_orders.db")
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS deliveries;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    city TEXT
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
    placed_at TEXT NOT NULL,
    payment_method TEXT NOT NULL,
    order_value REAL NOT NULL,
    status TEXT NOT NULL
);
CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL REFERENCES orders(order_id),
    product_name TEXT NOT NULL,
    perishable INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    price REAL NOT NULL,
    delivered INTEGER NOT NULL DEFAULT 1
);
CREATE TABLE deliveries (
    order_id INTEGER PRIMARY KEY REFERENCES orders(order_id),
    estimated_at TEXT NOT NULL,
    delivered_at TEXT NOT NULL,
    weather_delay_mode INTEGER NOT NULL DEFAULT 0
);
""")

customers = [(1, "Aarav", "Delhi"), (2, "Diya", "Delhi"), (3, "Kabir", "Gurgaon"), (4, "Meera", "Noida"),
             (5, "Rohan", "Delhi"), (6, "Sana", "Gurgaon"), (7, "Vikram", "Noida"), (8, "Ishita", "Delhi")]
cur.executemany("INSERT INTO customers VALUES (?,?,?)", customers)

# (product name, perishable 1/0, price)
products = [("Milk 1L", 1, 30), ("Yogurt 400g", 1, 40), ("Bananas 1kg", 1, 60), ("Tomatoes 1kg", 1, 45),
            ("Biscuits", 0, 25), ("Noodles", 0, 15), ("Rice 5kg", 0, 320), ("Shampoo", 0, 180)]
payments = ["UPI", "UPI", "card", "COD", "wallet", "netbanking"]
delays = [0, 0, 5, 10, 20, 25, 35, 45, 70]     # minutes after the estimate

start = datetime(2026, 9, 1, 9, 0)
item_id = 1
for order_id in range(1001, 1041):
    placed = start + timedelta(minutes=random.randint(0, 60 * 24 * 25))
    items, total = [], 0
    for name, perishable, price in random.sample(products, random.randint(1, 4)):
        qty = random.randint(1, 3)
        delivered = 0 if random.random() < 0.08 else 1
        items.append((item_id, order_id, name, perishable, qty, price, delivered))
        item_id += 1
        total += qty * price

    cur.execute("INSERT INTO orders VALUES (?,?,?,?,?,?)",
                (order_id, random.randint(1, 8), placed.isoformat(" ", "minutes"),
                 random.choice(payments), total, "delivered"))
    cur.executemany("INSERT INTO order_items VALUES (?,?,?,?,?,?,?)", items)

    estimated = placed + timedelta(minutes=15)
    delivered_at = estimated + timedelta(minutes=random.choice(delays))
    cur.execute("INSERT INTO deliveries VALUES (?,?,?,?)",
                (order_id, estimated.isoformat(" ", "minutes"), delivered_at.isoformat(" ", "minutes"),
                 1 if random.random() < 0.1 else 0))

# Make order 1001 a known test case: 35 minutes late, first item missing, no weather mode
cur.execute("""UPDATE deliveries
               SET delivered_at = strftime('%Y-%m-%d %H:%M', datetime(estimated_at, '+35 minutes')),
                   weather_delay_mode = 0
               WHERE order_id = 1001""")
cur.execute("""UPDATE order_items SET delivered = 0
               WHERE item_id = (SELECT MIN(item_id) FROM order_items WHERE order_id = 1001)""")
conn.commit()
print("Database ready")

Database ready


In [27]:
import pandas as pd

print(pd.read_sql("""
SELECT o.order_id, c.name, o.order_value,
       ROUND((julianday(d.delivered_at) - julianday(d.estimated_at)) * 1440) AS delay_minutes,
       d.weather_delay_mode
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
JOIN deliveries d ON d.order_id = o.order_id
WHERE o.order_id = 1001
""", conn))

print(pd.read_sql("SELECT product_name, quantity, price, delivered FROM order_items WHERE order_id = 1001", conn))

print(pd.read_sql("""
SELECT o.order_id, c.name, o.order_value,
       ROUND((julianday(d.delivered_at) - julianday(d.estimated_at)) * 1440) AS delay_minutes,
       d.weather_delay_mode
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
JOIN deliveries d ON d.order_id = o.order_id
WHERE delay_minutes > 30 AND d.weather_delay_mode = 0
ORDER BY delay_minutes DESC
""", conn))

   order_id  name  order_value  delay_minutes  weather_delay_mode
0      1001  Diya         25.0           35.0                   0
  product_name  quantity  price  delivered
0     Biscuits         1   25.0          0
    order_id    name  order_value  delay_minutes  weather_delay_mode
0       1016  Ishita        110.0           70.0                   0
1       1019    Diya        680.0           70.0                   0
2       1026    Diya        930.0           70.0                   0
3       1034   Aarav        575.0           70.0                   0
4       1008    Sana        380.0           45.0                   0
5       1013   Kabir        205.0           45.0                   0
6       1014  Ishita       1155.0           45.0                   0
7       1024    Diya        115.0           45.0                   0
8       1028   Rohan         60.0           45.0                   0
9       1040  Ishita       1005.0           45.0                   0
10      1001    Diya   

In [28]:
def lookup_order(order_id):
    cur = conn.cursor()
    order = cur.execute("""
        SELECT o.order_id, c.name, o.payment_method, o.order_value,
               ROUND((julianday(d.delivered_at) - julianday(d.estimated_at)) * 1440) AS delay_minutes,
               d.weather_delay_mode
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        JOIN deliveries d ON d.order_id = o.order_id
        WHERE o.order_id = ?
    """, (order_id,)).fetchone()

    if order is None:
        return {"error": f"No order found with id {order_id}"}

    items = cur.execute("""
        SELECT product_name, perishable, quantity, price, delivered
        FROM order_items WHERE order_id = ?
    """, (order_id,)).fetchall()

    return {
        "order_id": order[0],
        "customer": order[1],
        "payment_method": order[2],
        "order_value": order[3],
        "delay_minutes": int(order[4]),
        "weather_delay_mode": bool(order[5]),
        "items": [
            {"product": i[0], "perishable": bool(i[1]), "quantity": i[2],
             "price": i[3], "delivered": bool(i[4])}
            for i in items
        ],
    }

print(lookup_order(1001))
print(lookup_order(9999))

{'order_id': 1001, 'customer': 'Diya', 'payment_method': 'netbanking', 'order_value': 25.0, 'delay_minutes': 35, 'weather_delay_mode': False, 'items': [{'product': 'Biscuits', 'perishable': False, 'quantity': 1, 'price': 25.0, 'delivered': False}]}
{'error': 'No order found with id 9999'}


In [29]:
def late_delivery_credit(delay_minutes, weather_delay_mode):
    """Return the Rs. wallet credit owed for a late order, per the Late Delivery Policy."""
    if weather_delay_mode:
        return 0
    elif delay_minutes <= 15:
        return 0
    elif delay_minutes <= 30:
        return 30
    else:
        return 75

In [30]:
assert late_delivery_credit(10, False) == 0
assert late_delivery_credit(15, False) == 0
assert late_delivery_credit(16, False) == 30
assert late_delivery_credit(30, False) == 30
assert late_delivery_credit(31, False) == 75
assert late_delivery_credit(60, False) == 75
assert late_delivery_credit(61, False) == 75
assert late_delivery_credit(45, True) == 0
print("All tests passed")

All tests passed


In [31]:
REFUND_TIMELINE = {
    "wallet": "QuickKart Wallet, instant (within 5 minutes)",
    "UPI": "back to the UPI account in 2 to 3 working days",
    "card": "back to the card in 5 to 7 working days",
    "netbanking": "back to the bank account in 5 to 7 working days",
    "COD": "QuickKart Wallet within 24 hours, because no money was paid online",
}

In [32]:
def missing_item_refund(order_id):
    """Refund owed for items that were not delivered, and how the money comes back."""
    order = lookup_order(order_id)
    if "error" in order:
        return order

    missing = [i for i in order["items"] if not i["delivered"]]
    amount = sum(i["price"] * i["quantity"] for i in missing)

    return {
        "order_id": order_id,
        "missing_items": [i["product"] for i in missing],
        "refund_amount": amount,
        "how_and_when": REFUND_TIMELINE[order["payment_method"]],
        "delivery_and_handling_fees_also_refunded": len(missing) == len(order["items"]),
    }

print(missing_item_refund(1001))

{'order_id': 1001, 'missing_items': ['Biscuits'], 'refund_amount': 25.0, 'how_and_when': 'back to the bank account in 5 to 7 working days', 'delivery_and_handling_fees_also_refunded': True}


In [33]:
r = missing_item_refund(1001)
expected = conn.execute(
    "SELECT SUM(price * quantity) FROM order_items WHERE order_id = 1001 AND delivered = 0"
).fetchone()[0]
assert r["refund_amount"] == expected
assert "error" in missing_item_refund(9999)

clean = conn.execute("""
    SELECT order_id FROM orders
    WHERE order_id NOT IN (SELECT order_id FROM order_items WHERE delivered = 0)
    LIMIT 1
""").fetchone()[0]
assert missing_item_refund(clean)["refund_amount"] == 0
print("All tests passed")

All tests passed


In [34]:
def search_policy(question, k=5):
    """Search the QuickKart policy documents. Returns the most relevant passages with their source files."""
    res = col.query(query_texts=[question], n_results=k)
    return [{"source": m["source"], "text": t}
            for m, t in zip(res["metadatas"][0], res["documents"][0])]

print(search_policy("How long do I have to report a spoiled item?")[0])

{'source': 'item_issues_policy.txt', 'text': "SECTION 2: REPORTING WINDOWS\n2.1 Perishable items: report within 24 hours of delivery. Perishable means fruits, vegetables, dairy, bakery, meat, seafood and frozen food.\n2.2 Non-perishable items: report within 48 hours of delivery.\n2.3 The window is counted from the delivery time shown in the app, not from when you opened the package.\n2.4 Reports made after the window are reviewed only at QuickKart's discretion and may be declined.\n\nSECTION 3: EVIDENCE REQUIRED\n3.1 Missing items: no photo is needed. QuickKart checks the packing record instead.\n3.2 Wrong, damaged and spoiled items: one clear photo of the item and its packaging or label is required.\n3.3 Expired items: send a photo that clearly shows the expiry date on the label.\n3.4 If a photo is unclear, support may ask for one more photo within 12 hours of your report."}


In [35]:
import json

TOOLS = [
    {"type": "function", "function": {
        "name": "search_policy",
        "description": "Search the QuickKart policy documents (refunds, late delivery, cancellations, item issues, payments, fees, special cases). Use for any question about rules, timelines, fees, deadlines or conditions. Returns passages with their source file.",
        "parameters": {"type": "object",
                       "properties": {"question": {"type": "string", "description": "What to look up, in plain words"}},
                       "required": ["question"]}}},
    {"type": "function", "function": {
        "name": "lookup_order",
        "description": "Get the facts of one customer order from the orders database: payment method, order value, delay in minutes compared with the estimate, whether Weather Delay Mode was on, and each item with delivered true or false (delivered false means the item was missing). Use whenever the customer mentions an order id.",
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "integer"}},
                       "required": ["order_id"]}}},
    {"type": "function", "function": {
        "name": "late_delivery_credit",
        "description": "Calculator: the Rs. wallet credit owed for a late delivered order under the Late Delivery Policy. Always use this instead of working out the amount yourself. Take delay_minutes and weather_delay_mode from lookup_order.",
        "parameters": {"type": "object",
                       "properties": {"delay_minutes": {"type": "integer"},
                                      "weather_delay_mode": {"type": "boolean"}},
                       "required": ["delay_minutes", "weather_delay_mode"]}}},
    {"type": "function", "function": {
        "name": "missing_item_refund",
        "description": "Calculator: the refund amount for items that were not delivered in an order, which items they were, how and when the money comes back, and whether delivery and handling fees are also refunded. Use when an item is missing.",
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "integer"}},
                       "required": ["order_id"]}}},
]

TOOLS_BY_NAME = {
    "search_policy": search_policy,
    "lookup_order": lookup_order,
    "late_delivery_credit": late_delivery_credit,
    "missing_item_refund": missing_item_refund,
}

In [46]:
def agent(question, max_steps=6, max_searches=2, verbose=True):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    searches = 0

    for step in range(max_steps):
        last_chance = (step == max_steps - 1)
        response = groq_client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS,
            tool_choice="none" if last_chance else "auto", temperature=0)
        msg = response.choices[0].message

        if not msg.tool_calls:
            if msg.content and msg.content.strip():
                return msg.content
            messages.append({"role": "user", "content":
                "Please write your final answer to the customer now, using the tool results above."})
            continue

        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [{"id": tc.id, "type": "function",
                                         "function": {"name": tc.function.name,
                                                      "arguments": tc.function.arguments}}
                                        for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            if verbose:
                print(f"  -> {name}({args})")

            if name == "search_policy":
                searches += 1
            if name == "search_policy" and searches > max_searches:
                result = {"error": "Search limit reached. Do not search again. Answer using what you already have, or reply exactly: I don't know based on the policy documents."}
                if verbose:
                    print("     (search limit reached)")
            else:
                try:
                    result = TOOLS_BY_NAME[name](**args)
                except Exception as e:
                    result = {"error": str(e)}

            if name == "lookup_order" and isinstance(result, dict) and "error" in result:
              return f"I couldn't find order {args.get('order_id')}. Could you double-check the order number?"

            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result)})
    return "Stopped: too many steps."

In [53]:
def late_credit_tool(delay_minutes, weather_delay_mode):
    credit = late_delivery_credit(delay_minutes, weather_delay_mode)
    if weather_delay_mode:
        note = "Weather Delay Mode was on, so no late-delivery compensation is paid (Late Delivery Policy, clause 3.1)."
    elif credit == 0:
        note = "The delay is within the 15-minute grace period, so no compensation is paid (Late Delivery Policy, clause 2.1)."
    else:
        note = "Credited to QuickKart Wallet automatically within 30 minutes after delivery (Late Delivery Policy, clause 2.5). Compensation credits expire after 90 days."
    return {"credit_rs": credit, "note": note}

TOOLS_BY_NAME["late_delivery_credit"] = late_credit_tool

SYSTEM = """You are a customer support agent for QuickKart.
- Use lookup_order for facts about a specific order. Use search_policy for rules.
- Never work out credits or refund amounts yourself: use late_delivery_credit and missing_item_refund.
- State only facts that appear in a tool result. Do not invent policy names, timings or conditions. If you need a timing or condition, call search_policy first.
- Delivery and handling fees are refunded only if a tool result says so.
- If a rule differs by case (for example perishable and non-perishable items) and the customer does not say which case applies, give every case.
- If lookup_order returns an error, say you could not find that order id and ask the customer to check the number. Do not use the "I don't know" sentence for this.
- Call search_policy at most 2 times per question.
- Mention which policy file or tool each answer came from.
- If the tools do not give the answer, reply exactly: "I don't know based on the policy documents."
"""

In [45]:
print(agent("My order 1001 came late and an item was missing. What can I claim?"))
print("=====")
print(agent("How long do I have to report a spoiled item?"))
print("=====")
print(agent("What happened with order 9999?"))
print("=====")
print(agent("Can I change my delivery address after ordering?"))

  -> lookup_order({'order_id': 1001})
  -> late_delivery_credit({'delay_minutes': 35, 'weather_delay_mode': False})
  -> missing_item_refund({'order_id': 1001})
**What you can claim for Order 1001**

| Issue | Claim | How it will be processed | Policy reference |
|-------|-------|--------------------------|------------------|
| Late delivery (35 min delay, no weather‑delay mode) | **₹75 credit** to your QuickKart Wallet | Credited automatically within 30 minutes after delivery (Late Delivery Policy, clause 2.5). The credit expires after 90 days. | `late_delivery_credit` tool |
| Missing item (Biscuits) | **₹25 refund** to your original bank account | Refund will be credited back to the bank in 5–7 working days. Delivery and handling fees are also refunded. | `missing_item_refund` tool |

So you will receive a ₹75 wallet credit and a ₹25 refund to your bank account, with the delivery and handling fees also refunded.
=====
  -> search_policy({'question': 'How long do I have to report a s

In [52]:
import re, time
import pandas as pd

D = "ROUND((julianday(d.delivered_at) - julianday(d.estimated_at)) * 1440)"

def pick(where):
    row = conn.execute(f"""
        SELECT o.order_id FROM orders o JOIN deliveries d ON d.order_id = o.order_id
        WHERE {where} AND o.order_id != 1001 ORDER BY o.order_id LIMIT 1""").fetchone()
    return row[0] if row else None

mid_late     = pick(f"{D} BETWEEN 16 AND 30 AND d.weather_delay_mode = 0")
very_late    = pick(f"{D} > 30 AND d.weather_delay_mode = 0")
weather_late = pick(f"{D} > 15 AND d.weather_delay_mode = 1")
on_time      = pick(f"{D} <= 15 AND d.weather_delay_mode = 0")

def amt(n):    # matches Rs. 30, ₹30, INR 30 but not "within 30 minutes"
    return rf"(?:rs\.?|₹|inr)\s*\**\s*{n}(?:\.0+)?\b"

tests = []
def add(question, patterns, kind):
    tests.append({"question": question, "patterns": patterns, "kind": kind})

if mid_late:     add(f"Order {mid_late} arrived late. What compensation do I get?", [amt(30)], "late credit")
if very_late:    add(f"My order {very_late} was very late. Do I get any compensation?", [amt(75)], "late credit")
if weather_late: add(f"Order {weather_late} was late. Do I get compensation?", [r"weather"], "weather condition")
if on_time:      add(f"Order {on_time} arrived a bit after the estimate. Do I get late compensation?", [], "no credit (read it)")

for oid in [1004, 1016]:
    r = missing_item_refund(oid)
    add(f"Order {oid} had an item missing. What refund do I get, and are the delivery fees refunded too?",
        [amt(int(r["refund_amount"]))], "partial missing (fees not refunded)")

r = missing_item_refund(1001)
add("Order 1001 came late and an item was missing. What can I claim?",
    [amt(int(r["refund_amount"])), amt(75)], "late + missing")
add("What happened with order 9999?", [], "order not found (read it)")

add("How long do I have to report a problem with fresh milk compared with packaged noodles?", [r"\b24\b", r"\b48\b"], "policy")
add("I cancelled my order 10 minutes after placing it, before pickup. What will I pay?", [amt(20)], "policy")
add("I scheduled a delivery and want to cancel 30 minutes before the slot starts. What will it cost?", [amt(40)], "policy")
add("Can I pay cash on delivery for a Rs. 2,500 order?", [r"2,?000"], "policy")

add("Does QuickKart sell alcohol or cigarettes?", [r"don['’]t know"], "unanswerable")
add("Is there a paid membership that gives unlimited free delivery?", [r"don['’]t know"], "unanswerable")

rows = []
for t in tests:
    try:
        reply = agent(t["question"], verbose=False)
    except Exception as e:
        print("Stopped:", str(e)[:150]); break
    ok = all(re.search(p, reply, re.I) for p in t["patterns"]) if t["patterns"] else "read it"
    rows.append({"kind": t["kind"], "question": t["question"], "auto_check": ok, "answer": reply})
    time.sleep(5)

agent_df = pd.DataFrame(rows)
agent_df.to_csv("agent_results.csv", index=False)
for _, r in agent_df.iterrows():
    print(r["kind"], "|", r["question"]); print("AUTO:", r["auto_check"]); print(r["answer"]); print("-----")

late credit | Order 1004 arrived late. What compensation do I get?
AUTO: True
You’ll receive a **₹30 credit** to your QuickKart wallet.  
The credit is added automatically within 30 minutes after delivery and will expire after 90 days.  
*(Late Delivery Policy – clause 2.5)*
-----
late credit | My order 1005 was very late. Do I get any compensation?
AUTO: True
Your order was delayed by 35 minutes, so you’re eligible for a late‑delivery credit.  
According to the Late Delivery Policy, you will receive **₹75** credited to your QuickKart Wallet automatically within 30 minutes after delivery. The credit expires after 90 days. (Result from the `late_delivery_credit` tool.)
-----
weather condition | Order 1021 was late. Do I get compensation?
AUTO: True
Order 1021 was delivered 25 minutes late, but because the delay was covered under Weather Delay Mode, the Late Delivery Policy states that no wallet credit is awarded in this case. The credit calculation returned 0 ₹ (Late Delivery Policy, cl